In [ ]:
import os
import pandas as pd
pd.set_option('display.max_rows', None)  # Show all rows in DataFrame

In [ ]:
# Read all measured rim time files
# and merge them into a single DataFrame
directory = '../../../data/times/measured'
all_files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
df_list = []
for file in all_files:
    # rim filter since shot and rim times are in the same directory
    if 'rim' in file:
        df = pd.read_csv(file)
        df_list.append(df)

measured_times = pd.concat(df_list, ignore_index=True)
measured_times.drop(columns=['REGISTERED_SHOT_TIME'], inplace=True)

In [ ]:
measured_times.head()

## **Automated Rim Moment Detection Analysis**

In [ ]:
# Read all detected rim time files
# and merge them into a single DataFrame

directory = '../../../data/times/detected/'
all_files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
df_list = []
for file in all_files:
    # rim filter since shot and rim times are in the same directory
    if 'rim' in file:
        df = pd.read_csv(file)
        df_list.append(df)
detected_times = pd.concat(df_list, ignore_index=True)

detected_times.shape

In [ ]:
# Check percentage of NaN values for the DETECTED_RIM_TIME column
nan_percentage = detected_times['DETECTED_RIM_TIME'].isna().mean() * 100
print(f"Percentage of NaN values in DETECTED_RIM_TIME: {nan_percentage:.2f}%")

In [ ]:
detected_times.head()

In [ ]:
# Get one row for each EVENT_NUM and GAME_ID value combination
detected_times = detected_times.groupby(['EVENT_NUM', 'GAME_ID']).first().reset_index()

# Keep only certain columns
detected_times = detected_times[['GAME_ID','EVENT_NUM', 'DETECTED_RIM_TIME', 'DETECTED_SHOT_TIME']]

# Merge with measured_times on EVENT_NUM and GAME_ID
merged_mode = pd.merge(measured_times, detected_times, on=['EVENT_NUM', 'GAME_ID'])

# Remove rows where MEASURED_RIM_TIME is NaN
merged_mode = merged_mode.dropna(subset=['MEASURED_RIM_TIME'])

# Rows eligible for detection: present in the pipeline with a valid manual label
n_eligible = len(merged_mode)

merged_mode = merged_mode.dropna(subset=['DETECTED_RIM_TIME'])

n_detected = len(merged_mode)
coverage_pct = 100 * n_detected / n_eligible
print(f"Detection coverage: {n_detected}/{n_eligible} eligible situations got a DETECTED_RIM_TIME ({coverage_pct:.2f}%)")
print(f"NOTE: {n_eligible - n_detected} eligible situations ({100 - coverage_pct:.2f}%) had no detected rim time "
      "and are excluded from the accuracy statistics below.")

# variability analysis
differences = (merged_mode['DETECTED_RIM_TIME'] - merged_mode['MEASURED_RIM_TIME']).abs()
avg_difference_mode = differences.mean()
std_difference_mode = differences.std()
median_difference = differences.median()
q25, q75 = differences.quantile([0.25, 0.75])

print(f"Average difference between detected and measured rim times: {avg_difference_mode:.3f} ± {std_difference_mode:.3f} seconds")
print(f"Median difference: {median_difference:.3f} seconds")
print(f"IQR: [{q25:.3f}, {q75:.3f}] seconds")

# Sort by biggest difference
merged_mode['DIFFERENCE'] = merged_mode['DETECTED_RIM_TIME'] - merged_mode['MEASURED_RIM_TIME']
merged_mode = merged_mode.sort_values(by='DIFFERENCE', ascending=False)

merged_mode.head()

In [ ]:
merged_mode.tail()

In [ ]:
# Percentage of shots with difference less than 0.2, 0.4, 1.0 seconds
def percentage_within_threshold(merged_mode, threshold):
    within_threshold = merged_mode[merged_mode['DIFFERENCE'].abs() < threshold]
    percentage = (len(within_threshold) / len(merged_mode)) * 100
    print(f"Percentage of shots with difference less than {threshold} seconds: {percentage:.2f}%")

for i in [0.2, 0.4, 1.0]:
    percentage_within_threshold(merged_mode, i)

In [ ]:
# Top 10 worst shots (consider absolute difference)
worst_shots = merged_mode.copy()
worst_shots['ABSOLUTE_DIFFERENCE'] = worst_shots['DIFFERENCE'].abs()
worst_shots = worst_shots[['GAME_ID', 'EVENT_NUM', 'ABSOLUTE_DIFFERENCE']]
worst_shots = worst_shots.sort_values(by='ABSOLUTE_DIFFERENCE', ascending=False)
worst_shots.head(10)